In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import plotly.express as px
import geopandas as gpd
from pyclustering.cluster.kmedoids import kmedoids

print("Todas as bibliotecas carregadas com sucesso!")

# ==============================================================================
# PASSO 1: LIMPEZA E PREPARAÇÃO DOS DADOS
# ==============================================================================
print("\n[FASE 1] Carregando e limpando os arquivos do IBGE...")

PASTA_DADOS = 'dados_brutos'
caminho_pib = os.path.join(PASTA_DADOS, 'pib.xlsx')
caminho_alfabetizacao = os.path.join(PASTA_DADOS, 'alfabetizacao.xlsx')
caminho_densidade = os.path.join(PASTA_DADOS, 'densidade.xlsx')

df_pib = pd.read_excel(caminho_pib, skiprows=4, header=None, engine='openpyxl').iloc[1:, [0, -1]]
df_pib.columns = ['codigo_ibge', 'pib_per_capita']
df_pib = df_pib.dropna()
df_pib['codigo_ibge'] = pd.to_numeric(df_pib['codigo_ibge'], errors='coerce')
df_pib['pib_per_capita'] = pd.to_numeric(df_pib['pib_per_capita'], errors='coerce')
df_pib = df_pib.dropna().astype({'codigo_ibge': 'int64'})

df_alfabetizacao = pd.read_excel(caminho_alfabetizacao, skiprows=4, header=None, engine='openpyxl').iloc[1:, [0, -1]]
df_alfabetizacao.columns = ['codigo_ibge', 'taxa_alfabetizacao']
df_alfabetizacao = df_alfabetizacao.dropna()
df_alfabetizacao['codigo_ibge'] = pd.to_numeric(df_alfabetizacao['codigo_ibge'], errors='coerce')
df_alfabetizacao['taxa_alfabetizacao'] = pd.to_numeric(df_alfabetizacao['taxa_alfabetizacao'], errors='coerce')
df_alfabetizacao = df_alfabetizacao.dropna().astype({'codigo_ibge': 'int64'})

df_densidade = pd.read_excel(caminho_densidade, skiprows=4, header=None, engine='openpyxl').iloc[1:, [0, -1]]
df_densidade.columns = ['codigo_ibge', 'densidade_demografica']
df_densidade = df_densidade.dropna()
df_densidade['codigo_ibge'] = pd.to_numeric(df_densidade['codigo_ibge'], errors='coerce')
df_densidade['densidade_demografica'] = pd.to_numeric(df_densidade['densidade_demografica'], errors='coerce')
df_densidade = df_densidade.dropna().astype({'codigo_ibge': 'int64'})

print("\nUnificando os 3 datasets por código IBGE...")
df_final = pd.merge(df_pib, df_alfabetizacao, on='codigo_ibge', how='inner')
df_final = pd.merge(df_final, df_densidade, on='codigo_ibge', how='inner')

print(f"\nSUCESSO! Dataset consolidado com {len(df_final)} municípios.")
display(df_final.head())

In [ ]:
# ==============================================================================
# PASSO 1.5: ANÁLISE E TRATAMENTO DE OUTLIERS
# ==============================================================================
print("\n[FASE 1.5] Análise e tratamento de outliers (Método IQR)...")
df_antes = df_final.copy()
indicadores_analise = ['pib_per_capita', 'densidade_demografica']

for col in indicadores_analise:
    Q1 = df_final[col].quantile(0.25)
    Q3 = df_final[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_superior = Q3 + 1.5 * IQR

    outliers = df_final[df_final[col] > limite_superior]
    if not outliers.empty:
        print(f"Encontrados {len(outliers)} outliers para '{col}' (valores acima de {limite_superior:.2f})")
        df_final = df_final[df_final[col] <= limite_superior]

print(f"\nTotal de municípios antes da remoção de outliers: {len(df_antes)}")
print(f"Total de municípios após a remoção de outliers: {len(df_final)}")
print(f"Foram removidos {len(df_antes) - len(df_final)} municípios considerados extremos.")

In [ ]:
# ==============================================================================
# PASSO 2: NORMALIZAÇÃO
# ==============================================================================
print("\n[FASE 2] Normalizando os dados...")
indicadores = ['pib_per_capita', 'taxa_alfabetizacao', 'densidade_demografica']
scaler = StandardScaler()
df_normalizado = pd.DataFrame(
    scaler.fit_transform(df_final[indicadores]), 
    columns=indicadores,
    index=df_final.index
)
print("Dados normalizados com sucesso (média=0, desvio padrão=1).")

In [ ]:
# ==============================================================================
# PASSO 3: DETERMINAÇÃO ÓTIMA DO NÚMERO DE CLUSTERS (K)
# ==============================================================================
print("\n[FASE 3] Determinando K ótimo (Análise de Silhueta no dataset completo)...")
print("Aviso: Esta célula pode demorar vários minutos para executar.")

silhuetas = []
k_range = range(2, 11)

for k in k_range:
    np.random.seed(42)
    initial_medoids = np.random.choice(len(df_normalizado), k, replace=False).tolist()
    # A distância Euclidiana é o padrão, não precisa ser especificada
    kmedoids_instance = kmedoids(df_normalizado.values, initial_medoids)
    kmedoids_instance.process()
    
    cluster_indices = kmedoids_instance.get_clusters()
    labels = np.zeros(len(df_normalizado))
    for cluster_id, indices in enumerate(cluster_indices):
        labels[indices] = cluster_id
        
    silhuetas.append(silhouette_score(df_normalizado, labels))

# Mostra o gráfico da Silhueta para a tomada de decisão
fig_silhueta = px.bar(x=list(k_range), y=silhuetas,
                      title='<b>Coeficiente de Silhueta por K</b>',
                      labels={'x': 'Número de Clusters (K)', 'y': 'Silhueta'})
fig_silhueta.show()

K_ESCOLHIDO = 4
print(f"K escolhido = {K_ESCOLHIDO} clusters (baseado na análise de Silhueta).")

In [ ]:
# ==============================================================================
# PASSO 4: ALGORITMO K-MEDOIDS E NOMENCLATURA
# ==============================================================================
print(f"\n[FASE 4] Segmentando os municípios com K={K_ESCOLHIDO}...")
np.random.seed(42)
initial_medoids = np.random.choice(len(df_normalizado), K_ESCOLHIDO, replace=False).tolist()
modelo_final = kmedoids(df_normalizado.values, initial_medoids)
modelo_final.process()

# Atribui os clusters ao dataframe final
cluster_indices = modelo_final.get_clusters()
clusters = np.zeros(len(df_normalizado))
for cluster_id, indices in enumerate(cluster_indices):
    clusters[indices] = cluster_id
df_final['cluster_num'] = clusters.astype(int)

# Ordena e nomeia os perfis
perfil_para_ordenar = df_final.groupby('cluster_num')[indicadores].mean().round(2)
perfil_ordenado = perfil_para_ordenar.sort_values('pib_per_capita')

nomes_perfis = [
    "Vulneráveis",
    "Emergentes",
    "Consolidados",
    "Dinâmicos"
]
perfil_ordenado['Perfil'] = nomes_perfis

mapa_nomes = perfil_ordenado['Perfil'].to_dict()
df_final['perfil'] = df_final['cluster_num'].map(mapa_nomes)

# Identifica os medoides (pontos centrais)
indices_medoides = modelo_final.get_medoids()
df_final['medoids'] = df_final.index.isin(indices_medoides)

print("\nNomes definidos para cada cluster:")
print(mapa_nomes)
display(df_final.head())

In [ ]:
# ==============================================================================
# PASSO 4.5: ANÁLISE DOS FATORES DE DIFERENCIAÇÃO
# ==============================================================================
print("\n[FASE 4.5] Analisando os fatores mais importantes para a clusterização...")

# Obtém os valores normalizados dos medoides (centros)
centros_norm = df_normalizado.iloc[indices_medoides]
ranges = centros_norm.max() - centros_norm.min()
importancia = ranges.sort_values(ascending=False)

print("\nRanking de importância dos indicadores (maior range entre medoides):")
for i, (indicador, valor) in enumerate(importancia.items()):
    print(f"  {i+1}. {indicador.replace('_', ' ').title()}: {valor:.2f}")

fator_principal = importancia.index[0]
print(f"\nFator principal de diferenciação: {fator_principal.replace('_', ' ').title()}")

In [ ]:
# ==============================================================================
# PASSO 5: VISUALIZAÇÃO
# ==============================================================================
print("\n[FASE 5] Gerando gráficos para a apresentação...")

# --- GRÁFICO DE RADAR ---
df_radar_data = df_final.groupby('perfil')[indicadores].mean().reset_index()
df_radar_melted = pd.melt(df_radar_data, id_vars=['perfil'], var_name='Indicador', value_name='Valor')
fig_radar = px.line_polar(df_radar_melted, r='Valor', theta='Indicador', color='perfil',
                          line_close=True, title='<b>Personalidade de Cada Perfil Municipal (Valores Médios Reais)</b>',
                          template='plotly_white',
                          category_orders={'perfil': nomes_perfis})
fig_radar.show()

# --- BOXPLOTS (CORRIGIDO COM LOOP) ---
for indicador in indicadores:
    is_log_scale = (indicador != 'taxa_alfabetizacao')
    titulo = f'<b>Distribuição de {indicador.replace("_", " ").title()} por Perfil</b>'
    if is_log_scale:
        titulo += " (Escala Logarítmica)"

    fig_box = px.box(df_final, x='perfil', y=indicador, color='perfil',
                     title=titulo,
                     category_orders={'perfil': nomes_perfis},
                     log_y=is_log_scale)
    fig_box.show()

# --- MAPA DO BRASIL ---
print("\nGerando o Mapa do Brasil (pode levar um minuto)...")
try:
    url_mapa = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-100-mun.json"
    gdf_mapa = gpd.read_file(url_mapa)
    gdf_mapa['id'] = gdf_mapa['id'].astype('int64')
    mapa_final = gdf_mapa.merge(df_final, left_on='id', right_on='codigo_ibge', how='inner')

    fig_mapa = px.choropleth_mapbox(mapa_final, 
                                   geojson=mapa_final.geometry, 
                                   locations=mapa_final.index,
                                   color="perfil",
                                   mapbox_style="carto-positron",
                                   zoom=3.2, center = {"lat": -14.235, "lon": -51.925},
                                   opacity=0.7,
                                   title="<b>Distribuição dos Perfis de Municípios no Brasil</b>",
                                   category_orders={'perfil': nomes_perfis},
                                   labels={'perfil':'Perfil Municipal'})
    fig_mapa.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig_mapa.show()
except Exception as e:
    print(f"Falha ao gerar o mapa: {e}")

In [ ]:
# ==============================================================================
# PASSO 6: EXPORTAÇÃO E RELATÓRIO FINAL
# ==============================================================================
print("\n[FASE 6] Salvando resultados finais...")
PASTA_RESULTADOS = 'resultados'
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
print(f"Arquivos serão salvos em '{PASTA_RESULTADOS}/'")

# Exporta os arquivos
df_final.to_csv(os.path.join(PASTA_RESULTADOS, 'resultado_segmentacao_final.csv'), index=False)
perfil_ordenado.to_excel(os.path.join(PASTA_RESULTADOS, 'perfis_medios_clusters.xlsx'))
medoides_df = df_final[df_final['medoids']][['codigo_ibge', 'perfil'] + indicadores]
medoides_df.to_csv(os.path.join(PASTA_RESULTADOS, 'medoides_clusters.csv'), index=False)

# ==============================================================================
# RELATÓRIO FINAL
# ==============================================================================
print("\n" + "="*80)
print("RELATÓRIO EXECUTIVO DA SEGMENTAÇÃO DE MUNICÍPIOS")
print("="*80)

print(f"\n1. DADOS E METODOLOGIA:")
print(f"  - Foram analisados {len(df_antes)} municípios a partir de 3 indicadores do IBGE.")
print(f"  - Foram removidos {len(df_antes) - len(df_final)} municípios com dados extremos (outliers).")
print(f"  - O modelo de clusterização utilizado foi o K-Medoids com K={K_ESCOLHIDO}.")
print(f"  - A qualidade da separação dos clusters (Silhouette Score) foi de: {silhouette_score(df_normalizado, clusters):.3f}")

print(f"\n2. FATORES DE DIFERENCIAÇÃO:")
print(f"  - O indicador mais importante para separar os municípios foi:")
print(f"    - {importancia.index[0].replace('_', ' ').title()}")
print("  - Ranking de importância completo:")
for i, (indicador, valor) in enumerate(importancia.items()):
    print(f"    {i+1}. {indicador.replace('_', ' ').title()} (Range: {valor:.2f})")

print(f"\n3. PERFIS DOS CLUSTERS ENCONTRADOS:")
contagem_clusters = df_final['perfil'].value_counts().reindex(nomes_perfis)
perfil_ordenado_final = df_final.groupby('perfil')[indicadores].mean()

for perfil in nomes_perfis:
    num_municipios = contagem_clusters[perfil]
    media_pib = perfil_ordenado_final.loc[perfil, 'pib_per_capita']
    
    print(f"\n  - Perfil: {perfil} ({num_municipios} municípios - {num_municipios/len(df_final):.1%})")
    print(f"    - Característica: Municípios em estágio de desenvolvimento '{perfil}'.")
    print(f"    - PIB per capita médio: R$ {media_pib:,.2f}")

print(f"\n4. ARQUIVOS GERADOS:")
print(f"  - Os resultados detalhados foram salvos na pasta '{PASTA_RESULTADOS}/'")

print("\n" + "="*80)
print("PROJETO CONCLUÍDO | Análise pronta para apresentação!")
print("="*80)